# NLP Practical 1: Word Embeddings & Sequence Processing

This notebook merges the theory notes (`22_07_NLP_practice_1.md`) with the working implementation (`NLP_Practical_1.ipynb`), reordered into one logical walkthrough:

1. **Why `nn.Embedding` exists** (theory vs. practice, memory math)
2. **`nn.Embedding` as a lookup table** (toy demo)
3. **Loading pre-trained Word2Vec weights**
4. **Part 1 — Training & visualizing Armenian Word2Vec embeddings**
5. **The `batch_first=True` shape trap** (LSTM I/O shapes)
6. **Inside an LSTM cell** (cell state vs. hidden state, the 3 gates)
7. **Part 2 — Building `SentimentLSTM`** (bidirectional, multi-layer)
8. **IMDB data pipeline** (tokenizer, vocab, `DataLoader`)
9. **Training loop** (with validation + early stopping)
10. **Testing & inference** on unseen reviews

## 1. Why `nn.Embedding` Exists

### 1.1 Word2Vec. Տեսություն ընդդեմ Գործնականի

Որպեսզի հասկանանք, թե ինչու ենք օգտագործում ժամանակակից գործիքներ, պետք է նախ հասկանալ տեսական մոտեցման թերությունները:

**Տեսական մոտեցում (այն, ինչ սովորել ենք մինչ այս)**

Երբ մենք աշխատում ենք **Word2Vec**-ի հետ տեսականորեն, պրոցեսը հետևյալն է.

- Մենք վերցնում ենք բառերը որպես **one-hot վեկտորներ** (վեկտորներ, որտեղ միայն մեկ թիվն է 1, իսկ մնացած բոլորը՝ 0):
- Այդ վեկտորները բազմապատկում ենք **linear layer**-ով (գծային շերտով):
- Ստանում ենք **hidden layer**-ը (թաքնված շերտը) և այդպես ուսուցանում նեյրոնային ցանցը:

**Խնդիրը գործնականում**

Մաթեմատիկորեն այս մոտեցումը լիովին ճիշտ է։ Սակայն գործնականում առաջանում է շատ լուրջ արդյունավետության խնդիր:

Պատկերացրեք, որ ունեք 100,000 կամ 500,000 բառից բաղկացած բառարան։ Եթե դուք այդքան մեծ երկարությամբ **one-hot** վեկտորը բազմապատկեք **linear layer**-ով, համակարգը կկատարի միլիոնավոր (օրինակ՝ 100 միլիոն) գործողություններ: Սակայն քանի որ վեկտորի գրեթե բոլոր թվերը զրոներ են, այդ գործողությունների 99%-ը պարզապես անիմաստ տեղը զրոյով բազմապատկումներ են լինելու (ամեն բառի համար հազարավոր գործողություններ, որոնք արդյունքում տալիս են 0)։ Սա չափազանց ծանրաբեռնում և դանդաղեցնում է համակարգի աշխատանքը:

### 1.2 `nn.Embedding`-ը որպես Լուծում

Անընդհատ զրոների հետ բազմապատկելու խնդիրը շրջանցելու համար PyTorch-ը մեզ տրամադրում է **`nn.Embedding`** շերտը։ Մաթեմատիկորեն այն անում է ճիշտ նույն բանը, ինչ **linear layer**-ը, բայց խիստ օպտիմիզացված տարբերակով:

**Ինչո՞ւ մենք պարզապես խիտ (dense) Word2Vec վեկտորները միանգամից չենք փոխանցում մեր DataLoader-ով:**

Եթե ունենք ռեվյու, որը բաղկացած է 200 բառից, և յուրաքանչյուր բառ ներկայացված է 300 չափանի (300-dimensional) վեկտորով, այդքան շատ լողացող ստորակետով (floating-point) թվեր մեծ dataset-երի համար պահելը կխլի Գիգաբայթերով հիշողություն (RAM) և կտրուկ կդանդաղեցնի տվյալների տեղափոխումը դեպի GPU:

**The Scenario**

- **1 review** = 200 words
- **Each word** = 300-dimensional vector (300 floats)
- **Batch size** = 32 reviews (typical)
- **Dataset** = 100,000 reviews (modest)

If you pass dense vectors directly via the `DataLoader`:

```python
batch_shape = (32, 200, 300)  # [batch, seq_len, embed_dim]
```

**Memory per batch:** `32 × 200 × 300 × 4 bytes (float32) = 7,680,000 bytes ≈ 7.3 MB`

**Memory for full dataset (100k reviews):** `100,000 × 200 × 300 × 4 bytes = 24,000,000,000 bytes ≈ 24 GB RAM`

That's **just the embeddings** — not counting labels, metadata, `DataLoader` overhead, or the model itself. Most machines have 16–32 GB RAM. You'd OOM (out of memory) before training starts.

**The Alternative: Pass Integer IDs**

```python
batch_shape = (32, 200)  # [batch, seq_len] — each entry is an int (word ID)
```

**Memory per batch:** `32 × 200 × 4 bytes (int32) = 25,600 bytes ≈ 25 KB`

**Memory for full dataset:** `100,000 × 200 × 4 bytes = 80,000,000 bytes ≈ 80 MB RAM`

**Difference: 24 GB vs. 80 MB → 300× smaller.**

**What Happens on the GPU**

The `nn.Embedding` layer lives on the GPU as a **single lookup table**:

```
Embedding matrix: [vocab_size, 300]  →  e.g., [50,000, 300] = 60 MB (once)
```

When your batch of integer IDs `[32, 200]` hits the GPU:
1. `nn.Embedding` does an **index lookup** (instant, no math)
2. Returns the 300-d vector for each ID
3. Now you have `[32, 200, 300]` **on the GPU** — where compute happens

| Stage | Dense Vectors in DataLoader | Integer IDs in DataLoader |
|-------|----------------------------|---------------------------|
| **CPU RAM** | 24 GB (crashes) | 80 MB (tiny) |
| **CPU→GPU transfer** | 7.3 MB/batch (slow) | 25 KB/batch (instant) |
| **GPU memory** | Same either way | Same either way |
| **Flexibility** | Hard to change embeddings | Easy to swap/fine-tune |

> **The Key Insight:** Don't move heavy data through the pipeline. Move lightweight pointers (indices), expand them only where the compute happens (GPU).

This is exactly the two-step process:
1. **Tokenizer** → words become integer IDs (tiny, live on CPU)
2. **nn.Embedding** → IDs become dense vectors (heavy, live on GPU)

**Ինչպե՞ս է աշխատում գործընթացը (Երկու քայլանոց մոտեցում)**

1. **Tokenizer / Vocabulary (Բառարանի ստեղծում):** Ամբողջական վեկտորներ պահելու փոխարեն, մենք մեր dataset-ի յուրաքանչյուր ունիկալ բառին կցում ենք մեկ պարզ ամբողջ թիվ (integer ID): _Օրինակ՝ "the" → 1, "movie" → 402, "great" → 1895։_
2. **Lookup table (Որոնման աղյուսակ):** GPU-ի հիշողության մեջ **`nn.Embedding`**-ը ստեղծում է կշիռների մեծ մատրից (weight matrix), որն ունի հստակ չափողականություն՝ `[|V|, d]`, որտեղ `|V|`-ն բառարանի ընդհանուր չափն է, իսկ `d`-ն վեկտորի չափողականությունն է (dimensions):

Սա աշխատում է որպես նախապես կառուցված **dictionary**: Երբ դուք շերտին փոխանցում եք բառերի ID-ներ (օրինակ՝ [1, 402, 1895]), `nn.Embedding`-ը ոչ մի բազմապատկում չի անում: Այն ուղղակի O(1) ժամանակում մտնում է այդ աղյուսակ և վերադարձնում է 1, 402 և 1895 ինդեքսներով տողերը։

_Այս տողերի արժեքները հենց նեյրոնային ցանցի պարամետրերն են, որոնք ուսուցման պրոցեսում փոփոխվում և հարմարվում են **gradient descent**-ի միջոցով։_

## 2. `nn.Embedding` as a Lookup Table — Toy Demo

Ահա գործնական PyTorch կոդը, որը ցույց է տալիս `nn.Embedding`-ի աշխատանքը որպես Lookup Table.

- Բառարանի չափը (`vocab_size`) = 10 (այսինքն՝ ունենք ընդամենը 10 ունիկալ բառ)
- Vector-ի չափողականությունը (`embed_dim`) = 4

Այս դեպքում `nn.Embedding`-ը ստեղծում է 10×4 չափի մատրիցա, որտեղ ամեն տող համապատասխանում է մեկ բառի 4-չափանի վեկտորին։ Եթե ունենք նախադասություն, որի բառերի ID-ներն են `[1, 4, 2]` (օրինակ՝ _"the movie was"_), `nn.Embedding` շերտը կանչելիս այն չի կատարում մատրիցական բազմապատկումներ։ Այն պարզապես վերցնում է մատրիցայի **1-ին**, **4-րդ** և **2-րդ** տողերը և վերադարձնում որպես այդ նախադասության embed արված tensor:

In [ ]:
import torch
import torch.nn as nn
import numpy as np

# --- Demo: How nn.Embedding acts as a Lookup Table ---
vocab_size = 10
embed_dim = 4

# Initialize a random embedding layer
emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

# Inspect the underlying lookup matrix (Shape: [10, 4])
print("Embedding Weight Matrix:")
print(emb.weight.data)

# Pass integer token IDs (e.g., a batch of 2 sequences, each of length 3)
token_ids = torch.tensor([[1, 4, 2],
                          [0, 9, 3]])

# Lookup operation
embedded_output = emb(token_ids)
print("\nOutput Tensor Shape:", embedded_output.shape) # [batch_size, seq_len, embed_dim]
print("Retrieved Vector for Token ID 4 (Row 4):")
print(embedded_output[0, 1])

**Գործնական Կոդի Վերլուծություն**

- **`emb = nn.Embedding(10, 4)`:** Ստեղծում է 10×4 չափի պատահական պատրաստված (random initialized) weight մատրիցա։
- **`emb.weight.data`:** Ցույց է տալիս հենց այդ 10×4 աղյուսակը։ Յուրաքանչյուր ինդեքս (0-ից 9) ունի իրեն համապատասխան 4 թիվ պարունակող վեկտորը։
- **`token_ids` Tensor-ի չափը:** Փոխանցում ենք 2 նախադասություն (`batch_size = 2`), որոնցից յուրաքանչյուրն ունի 3 բառ (`seq_len = 3`): Tensor-ի չափն է `[2, 3]`:
- **`embedded_output` Tensor-ի չափը:** `emb(token_ids)` կանչելուց հետո ելքային tensor-ի չափը դառնում է `[2, 3, 4]` (տեսքը՝ `[batch_size, seq_len, embed_dim]`)։ Ամեն մի integer ID փոխարինվել է իր 4-չափանի վեկտորով։
- **Որոնման (Lookup) Ստուգում:** `embedded_output[0, 1]`-ը նշանակում է՝ առաջին նախադասության (`batch index 0`) երկրորդ բառը (`sequence index 1`), որի ID-ն 4 է։ Ելքում ստացված վեկտորը ճշգրտորեն համընկնում է `emb.weight`-ի 4-րդ տողի հետ։
- **`grad_fn=<SelectBackward0>`:** Սա ապացուցում է, որ ստացված վեկտորը կապված է PyTorch-ի computational graph-ի հետ։ Եթե model-ը սովորեցնենք, այս վեկտորի արժեքները ավտոմատ կթարմացվեն backpropagation-ի միջոցով։

## 3. Loading Pre-trained Word2Vec Weights into `nn.Embedding`

Եթե դուք արդեն ունեք մեկ այլ տեղից վերցված կամ նախապես սովորած Word2Vec վեկտորներ (օրինակ՝ Gensim գրադարանի միջոցով), ձեզ պարտադիր չէ զրոյից սովորեցնել ձեր **embeddings**-ները: Դուք կարող եք ուղիղ բեռնել այդ պատրաստի կշիռները PyTorch-ի `nn.Embedding` շերտի մեջ՝ օգտագործելով `nn.Embedding.from_pretrained()` ֆունկցիան:

```python
# Convert Gensim Word2Vec matrix to a PyTorch tensor
weights = torch.FloatTensor(w2v_model.wv.vectors)

# Ստեղծում ենք embedding layer-ը և սկզբնարժեքավորում պատրաստի Word2Vec կշիռներով
embedding_layer = nn.Embedding.from_pretrained(weights, freeze=False)
```

**Կարևոր պարամետր՝ `freeze`**

- **`freeze=True`:** Պահպանում է Word2Vec վեկտորները ստատիկ (սառեցված) վիճակում ողջ training-ի ժամանակ: Դրանց արժեքները երբեք չեն փոխվում:
- **`freeze=False`:** Թույլ է տալիս, որ **backpropagation**-ի (հետադարձ տարածման) ընթացքում նեյրոնային ցանցը փոփոխի (fine-tune անի) այդ Word2Vec վեկտորները՝ հատուկ ձեր կոնկրետ խնդրին (օրինակ՝ ձեր կոնկրետ **Sentiment Analysis**-ին) ավելի լավ հարմարեցնելու համար:

**End-to-End օպտիմիզացիայի առավելությունը**

`nn.Embedding` շերտը թույլ է տալիս, որ embedding-ների ստացման և օպտիմիզացիայի պրոցեսը միացնենք հիմնական մոդելի ուսուցմանը։ Պարտադիր չէ նախապես առանձին փուլով բոլոր տեքստերը դարձնել վեկտորներ․ PyTorch-ը embedding-ները դիտարկում է որպես մոդելի պարամետրեր, որոնք ունեն `grad_fn` և կարող են օպտիմիզացվել gradient descent-ի միջոցով հենց ուսուցման ընթացքում։

`self.embedding` creates a **PyTorch lookup layer**, but how it handles weights depends on which branch runs:

**1. If `pretrained_vectors` is provided:** `nn.Embedding.from_pretrained(...)` initializes the layer and populates its weights directly with your pre-trained Word2Vec vectors. `freeze=False` allows PyTorch to continue updating and fine-tuning these weights during backpropagation.

**2. If `pretrained_vectors` is `None`:** `nn.Embedding(vocab_size, embed_dim)` creates a layer with **randomly initialized weights** (PyTorch defaults) — a blank lookup table from scratch.

**What the Layer Actually Does at Runtime** — in both cases, `self.embedding(x)` acts as an index lookup table:

```
Input token IDs (x):     [ [12,  405,  89] ]  (Shape: [1, 3])
                                │
                                ▼
                       Look up rows 12, 405, 89
                                │
                                ▼
Output (embeds):         [ [ [v12], [v405], [v89] ] ]  (Shape: [1, 3, 100])
```

It converts integer token indices into dense float vectors of length `embed_dim`, creating the starting representation fed into the LSTM.

## 4. Part 1: Training & Visualizing Armenian Word2Vec Embeddings

Let's load a subset of Armenian Wikipedia, train a Word2Vec model on it, convert its vectors to a PyTorch `nn.Embedding` layer, and project the high-dimensional vectors to 2D space using Principal Component Analysis (PCA).

In [ ]:
# !pip install datasets
!pip install gensim

In [ ]:
import re
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# 1. Load 1% snapshot of Armenian Wikipedia
print("Downloading Armenian Wikipedia dump...")
dataset = load_dataset("wikimedia/wikipedia", "20231101.hy", split="train[:1%]")

# 2. Simple tokenizer for Armenian alphabet
def tokenize_am(text):
    text = text.lower()
    text = re.sub(r'[^ա-ֆև ]+', '', text)  # keep only Armenian letters and spaces
    return text.split()

print("Tokenizing corpus...")
corpus = [tokenize_am(article['text']) for article in dataset]

# 3. Train Word2Vec Skip-Gram model
print("Training Word2Vec model...")
w2v_model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=5, sg=1, workers=4)

# 4. Extract weights and load into PyTorch nn.Embedding
w2v_weights = torch.FloatTensor(w2v_model.wv.vectors)
nn_embedding_am = nn.Embedding.from_pretrained(w2v_weights, freeze=False)
print(f"Successfully created PyTorch Embedding layer with shape: {nn_embedding_am.weight.shape}")

# 5. Visualize select words using PCA
target_words = ['երևան', 'հայաստան', 'քաղաք', 'գյուղ', 'մարդ', 'կին', 'տղամարդ']
valid_words = [w for w in target_words if w in w2v_model.wv]
vectors = np.array([w2v_model.wv[w] for w in valid_words])

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], color='crimson')
for i, word in enumerate(valid_words):
    plt.annotate(word, (vectors_2d[i, 0] + 0.01, vectors_2d[i, 1] + 0.01), fontsize=13)
plt.title("2D PCA Projection of Armenian Word2Vec Vectors")
plt.grid(True)
plt.show()

`w2v_model.wv` stands for **Word Vectors**. It is a distinct object (specifically, a `KeyedVectors` instance) inside the Gensim Word2Vec model that stores the actual trained embeddings and the vocabulary dictionary.

- **Separation of Training vs. Inference:** The main `w2v_model` object contains all the heavy computational machinery needed for _training_ (hidden layers, learning rates, negative sampling logic). The `.wv` object acts purely as a lookup table for the finished word vectors.
- **Memory Efficiency:** Gensim separates these so that once training is complete, you can keep only the `.wv` object and discard the rest of the model to free up system memory.
- **The Code Context:** `w2v_model.wv.vectors` extracts the raw 2D NumPy array of all numeric weights (shape `[vocab_size, vector_size]`). The code converts this into a PyTorch tensor and injects it directly into the `nn.Embedding` layer.

Let's also try a broader set of country/geography words on the same trained model:

In [ ]:
target_words = ['հանրապետություն', 'երկիր', 'ռուսաստան', 'թուրքիա', 'ամն', 'ամերիկա', 'ուրուգվայ', 'իրան', 'ֆրանսիա', 'հայաստան', 'վրաստան', 'ադրբեջան']
valid_words = [w for w in target_words if w in w2v_model.wv]
vectors = np.array([w2v_model.wv[w] for w in valid_words])

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], color='crimson')
for i, word in enumerate(valid_words):
    plt.annotate(word, (vectors_2d[i, 0] + 0.01, vectors_2d[i, 1] + 0.01), fontsize=13)
plt.title("2D PCA Projection of Armenian Word2Vec Vectors")
plt.grid(True)
plt.show()

For more interesting embedding visualizations, check https://projector.tensorflow.org/.

## 5. The PyTorch Shape Trap (`batch_first=True`)

In Computer Vision, tensor shapes are almost universally `[batch_size, channels, height, width]`.

By default, PyTorch's `nn.LSTM` expects inputs in the shape `[seq_len, batch_size, embed_dim]`. Passing a standard batch of shape `[batch_size, seq_len, embed_dim]` without configuring the layer will lead to shape mismatch errors.

Always explicitly set `batch_first=True` when instantiating `nn.LSTM` to keep the batch size as the first dimension:

$$\text{Input Shape: } [N, T, d] \qquad \text{Output Shape: } [N, T, m]$$

Where $N$ is the batch size, $T$ is the sequence length, $d$ is the embedding dimension, and $m$ is the hidden state dimension.

`batch_first` is a boolean flag in PyTorch neural network layers (like `nn.LSTM` or `nn.Transformer`) that tells PyTorch **which dimension index represents the batch size** in your input and output tensors.

**The Two Tensor Layouts**

- **`batch_first=True`** (used in this notebook)
  - Tensor shape format: `[batch_size, seq_len, feature_dim]`
  - Example shape: `[16, 50, 100]`
  - Intuition: Index 0 is the batch item index, index 1 is the token position in the sequence, and index 2 is the embedding dimension.
- **`batch_first=False`** (PyTorch's default)
  - Tensor shape format: `[seq_len, batch_size, feature_dim]`
  - Example shape: `[50, 16, 100]`
  - Intuition: PyTorch processes time step by time step down the sequence, so placing `seq_len` first is slightly more efficient for low-level C++/CUDA memory access during recurrent loops.

**Why It Matters**

1. **Readability & Consistency:** Most modern Deep Learning libraries (Hugging Face `transformers`, TensorFlow/Keras) use `[batch_size, seq_len, embed_dim]`. Setting `batch_first=True` aligns PyTorch with standard data loaders.
2. **Indexing Behavior:** Because `batch_first=True` is used, `output[:, -1, :]` grabs the final time-step hidden state across all batch items. If `batch_first=False` were used, you'd instead slice with `output[-1, :, :]`.

_(Note: `batch_first` only affects the main sequence input and `output` tensors. Internal state tensors like `h_n` and `c_n` always retain the shape `[num_layers * num_directions, batch_size, hidden_dim]` regardless of this setting)._

In [ ]:
batch_size = 16
seq_len = 50
embed_dim = 100
hidden_dim = 256

# Initialize LSTM with batch_first=True
lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)

# Generate dummy embedded input: [batch_size, seq_len, embed_dim]
dummy_embeddings = torch.randn(batch_size, seq_len, embed_dim)

# Forward pass through LSTM
# output: contains hidden states for ALL time steps [batch_size, seq_len, hidden_dim]
# (h_n, c_n): contains final states [num_layers, batch_size, hidden_dim]
output, (h_n, c_n) = lstm(dummy_embeddings)

print("Full Sequence Output Shape (out):", output.shape)
print("Final Hidden State Shape (h_n):   ", h_n.shape)

# Extracting the last hidden state h_T for sentiment classification
# Method A: From output tensor
h_T_from_out = output[:, -1, :]  # [batch_size, hidden_dim]

# Method B: From h_n tuple
h_T_from_hn = h_n.squeeze(0)     # [batch_size, hidden_dim]

print("Extracted Final Vector Shape:    ", h_T_from_hn.shape)

Here is a visual step-by-step breakdown of how this exact code processes data inside PyTorch.

### 5.1 Data Shape Alignment

Before processing starts, PyTorch reads the incoming tensor dimensions based on `batch_first=True`:

```
dummy_embeddings: [ 16 ,  50 , 100 ]
                    │     │     │
                    │     │     └─► input_size (embed_dim): 100 numbers per word
                    │     └───────► seq_len: 50 words per sentence
                    └─────────────► batch_size: 16 sentences processed in parallel
```

### 5.2 Inner Recurrent Loop Visualized

The LSTM processes the sequence step-by-step across time $t=1$ to $t=50$. At every time step $t$, the cell takes **two inputs** (the current word embedding $x_t$ and the previous hidden state $h_{t-1}$) and produces **two outputs** (the new hidden state $h_t$ and updated cell memory $c_t$).

```
           ┌─────────────────────────────────────────┐
           │        LSTM Cell (Hidden Dim: 256)      │
           └─────────────────────────────────────────┘

 (Start) ──► [h_0, c_0] (Initialized to all zeros by PyTorch)
                 │
                 ▼
  x_1   ──► ┌─────────┐
[16,100]    │ Step 1  │ ──► h_1 [16, 256] ──┐
            └─────────┘                     │ Collected into 'output'
                 │                          │
                 ▼ (passes h_1, c_1)        │
  x_2   ──► ┌─────────┐                     │
[16,100]    │ Step 2  │ ──► h_2 [16, 256] ──┤
            └─────────┘                     │
                 │                          │
                ... (Steps 3 through 49)    │
                 │                          │
                 ▼                          │
  x_50  ──► ┌─────────┐                     │
[16,100]    │ Step 50 │ ──► h_50 [16, 256] ─┼──► Also stored as h_n
            └─────────┘                     │
                 │                          │
                 ▼                          ▼
            c_50 [16, 256]            output tensor
           (Stored as c_n)          [16, 50, 256]
```

### 5.3 Understanding the Return Values

The forward pass returns a tuple: `output, (h_n, c_n)`.

- **`output` `[16, 50, 256]`**: Stacks the hidden states from **every single time step** ($h_1, h_2, \dots, h_{50}$).
  - `output[:, 0, :]` holds $h_1$ (after word 1).
  - `output[:, 49, :]` holds $h_{50}$ (after word 50).
- **`h_n` `[1, 16, 256]`**: The **final short-term memory** (hidden state $h_{50}$) after reading the whole sequence. The first dimension is `1` because `num_layers=1`.
- **`c_n` `[1, 16, 256]`**: The **final long-term memory** (cell state $c_{50}$) at step 50.

### 5.4 Extracting $h_T$: Methods Compared

```
Method A (From output):  output[:, -1, :] ──► Shape: [16, 256] (Slices the last time step)
Method B (From h_n):     h_n.squeeze(0)   ──► Shape: [16, 256] (Removes layer dimension)
```

Both methods yield the exact same tensor values representing the final summary vector for each sequence in the batch.

## 6. Inside an LSTM Cell: Cell State vs. Hidden State

In an LSTM, the fundamental unit at each time step consists of two distinct vectors passed forward through time: the **Cell State ($c_t$)** and the **Hidden State ($h_t$)**. They act as two separate branches of memory designed to solve the vanishing gradient problem.

### 6.1 Memory Cell / Cell State ($c_t$)

- **Role:** Long-term memory (the "conveyor belt").
- **Behavior:** It runs straight down the entire chain with only minimal linear interactions. Information can flow down it unchanged for many time steps, preserving early sequence context across long distances.
- **How it changes:** Updated purely through **additive** and **multiplicative gates** rather than dense matrix multiplications, which prevents gradients from vanishing during backpropagation.

### 6.2 Hidden Cell / Hidden State ($h_t$)

- **Role:** Short-term memory & current output.
- **Behavior:** Contains a filtered, immediate representation of the sequence at time $t$. This is the vector used to calculate loss, make predictions at step $t$, or pass directly to the next layer/classifier.
- **How it changes:** Computed by taking the updated Cell State ($c_t$), running it through a non-linear activation ($\tanh$), and squeezing/filtering it through an **Output Gate**.

### 6.3 The 3 Gates: How They Interact

The interaction between $c_t$ and $h_t$ is controlled by three sigmoid-activated neural network layers ($\sigma$, outputting values between 0 and 1):

```
       Input (x_t) & Previous Hidden (h_{t-1})
                          │
       ┌──────────────────┼──────────────────┐
       │                  │                  │
       ▼                  ▼                  ▼
[ Forget Gate ]    [ Input Gate ]    [ Output Gate ]
  f_t = σ(...)       i_t = σ(...)      o_t = σ(...)
       │                  │                  │
       │  (1) Erase       │  (2) Add         │
       ▼                  ▼                  │
┌─────────────────────────────────────────┐  │
│ Cell State Update:                      │  │
│ c_t = (f_t * c_{t-1}) + (i_t * g_t)     │  │
└─────────────────────────────────────────┘  │
                   │                         │
                   │  (3) Filter             │
                   ▼                         ▼
┌──────────────────────────────────────────────┐
│ Hidden State Update:                         │
│ h_t = o_t * tanh(c_t)                        │
└──────────────────────────────────────────────┘
```

**1. Forget Gate ($f_t$):** Determines what fraction of the old long-term memory ($c_{t-1}$) to throw away.

$$\mathbf{f}_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$$

**2. Input Gate ($i_t$ & $\tilde{c}_t$):** Determines what new information from the current input ($x_t$) to add into the long-term memory ($c_t$).

$$\mathbf{c}_t = f_t \odot c_{t-1} + i_t \odot \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$$

**3. Output Gate ($o_t$):** Determines what parts of the long-term memory ($c_t$) are relevant _right now_ to expose in the short-term hidden state ($h_t$).

$$\mathbf{h}_t = o_t \odot \tanh(c_t)$$

### 6.4 Key Comparison

| Attribute | Cell State ($c_t$) | Hidden State ($h_t$) |
|---|---|---|
| **Type of Memory** | Long-term memory storage | Short-term working memory |
| **Modifications** | Modified only by addition/subtraction | Completely re-filtered at every step |
| **Exposed Outside** | Stays internal within the LSTM cell | Exposed as output to next layers/time steps |
| **Analogy** | A notebook holding raw facts | A speaker summarizing what was just read |

### 6.5 What is a Hidden State ($h_t$), Really?

A **hidden state** is simply a vector of numbers that serves as the network's **memory vector** at time step $t$.

- **Why "Hidden"?** It sits between the input layer ($x_t$) and the final output layer ($y_t$). It isn't directly observed in the raw input data; the network learns to calculate it internally.
- **What does it do?** As an RNN or LSTM reads a sequence item by item, it updates $h_t$ at each step. By time step $t$, $h_t$ holds a numeric summary of everything processed from step 1 up to step $t$.

**What Does $[h_{t-1}, x_t]$ Mean?**

In mathematical formulas, $[h_{t-1}, x_t]$ denotes **vector concatenation** (stacking two vectors end-to-end into one longer vector).

- **$x_t$**: The current input vector at time step $t$ (e.g., the word embedding, length $D_x$).
- **$h_{t-1}$**: The previous hidden state vector (memory accumulated up to the previous word, length $D_h$).

*Visualizing Concatenation*

```
Previous Hidden State (h_{t-1}):  [ 0.5, -0.2,  0.9 ]  (Size: 3)
Current Input Embedding (x_t):   [ 1.2,  0.0, -0.7,  0.4 ]  (Size: 4)

Concatenated [h_{t-1}, x_t]:      [ 0.5, -0.2,  0.9,  1.2,  0.0, -0.7,  0.4 ]  (Size: 3 + 4 = 7)
```

**Why Concatenate Them?**

This single concatenated vector is multiplied by a single weight matrix $W$ to calculate gate values:

$$W \cdot [h_{t-1}, x_t] + b$$

By concatenating them, the gate (Forget Gate, Input Gate, etc.) gets to look at **what you just read ($x_t$)** alongside **what you remember ($h_{t-1}$)** at the exact same time to decide what information to keep, throw away, or update.

## 7. Part 2: DIY Sentiment Analysis (IMDB Dataset)

Now let's build the `SentimentLSTM` model.

**Requirements**:
1. Initialize `nn.Embedding` (optionally load Word2Vec weights if provided).
2. Instantiate `nn.LSTM` with `batch_first=True`.
3. Extract the final hidden state $h_T$ corresponding to the last word of each review.
4. Pass $h_T$ into a linear layer to output binary classification logits.

This version adds **bidirectionality** and **stacked layers** on top of the minimal single-direction LSTM shown above, plus dropout for regularization.

In [ ]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, pretrained_vectors=None):
        super().__init__()

        if pretrained_vectors is not None:
            self.embedding = nn.Embedding.from_pretrained(pretrained_vectors, freeze=False)
        else:
            self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)

        # 1. Add Bidirectionality & Multiple Layers
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=2,           # Stacks 2 LSTMs
            bidirectional=True,     # Reads text forward AND backward
            batch_first=True,
            dropout=0.3             # Dropout between LSTM layers
        )

        # 2. Add Dropout Layer before classifier
        self.dropout = nn.Dropout(0.5)

        # 3. Adjust FC layer for Bidirectional output (hidden_dim * 2)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        # x shape: [batch_size, seq_len] (Integer Token IDs)
        # 1. Look up embeddings -> [batch_size, seq_len, embed_dim]
        embeds = self.embedding(x)

        # h_n shape: [num_layers * num_directions, batch_size, hidden_dim] -> [4, batch_size, hidden_dim]
        output, (h_n, c_n) = self.lstm(embeds)

        # Concatenate the final forward state (index -2) and backward state (index -1)
        h_forward = h_n[-2, :, :]
        h_backward = h_n[-1, :, :]
        h_T = torch.cat((h_forward, h_backward), dim=1)  # Shape: [batch_size, hidden_dim * 2]

        h_T = self.dropout(h_T)
        logits = self.fc(h_T).squeeze(1)

        return logits

# --- Sanity Check ---
test_vocab_size = 5000
test_model = SentimentLSTM(vocab_size=test_vocab_size, embed_dim=100, hidden_dim=128)
dummy_tokens = torch.randint(0, test_vocab_size, (8, 200))  # Batch of 8 reviews, length 200
test_logits = test_model(dummy_tokens)

print("Output Logits Shape:", test_logits.shape)  # Should be [8]
assert test_logits.shape == torch.Size([8]), "Shape check failed! Check your squeeze() operations."
print("Sanity Check Passed!")

### 7.1 Why `h_n[-2, :, :]` and `h_n[-1, :, :]`?

The indexing comes down to how PyTorch shapes and orders the `h_n` tensor inside an `nn.LSTM`.

Regardless of whether `batch_first` is `True` or `False`, PyTorch always formats `h_n` with this shape:

$$\text{Shape of } h_n = [\text{num\_layers} \times \text{num\_directions}, \text{batch\_size}, \text{hidden\_dim}]$$

When you set `num_layers = 2` and `bidirectional = True`:

- $\text{num\_directions} = 2$ (1 forward, 1 backward)
- $\text{total directions/layers} = 2 \times 2 = 4$

PyTorch stacks these layers sequentially along dimension `0` in this exact order:

```
Index 0: Layer 1 — Forward  LSTM
Index 1: Layer 1 — Backward LSTM
Index 2: Layer 2 — Forward  LSTM  <-- index -2 (Second to last)
Index 3: Layer 2 — Backward LSTM  <-- index -1 (Last)
```

So `h_n[-2]` and `h_n[-1]` are always the **final layer's** forward and backward hidden states — exactly what you want to concatenate for classification, since they've seen the whole sequence in both directions after being refined by the earlier layer.

### 7.2 The Bidirectional Math, Step by Step

Under the hood, a **Bidirectional LSTM** combines linear algebra (matrix multiplications) and non-linear gating functions to pass information forward and backward through sequence time steps.

**Step-by-Step Mathematical Flow** — for every time step $t$ in a sequence of length $T$:

```
 Forward Pass (Left to Right)                Backward Pass (Right to Left)
       t = 1 ──► 2 ──► ... ──► T                   T ──► ... ──► 2 ──► 1

   x_t + h_forward_{t-1}                       x_t + h_backward_{t+1}
            │                                           │
            ▼                                           ▼
  ┌───────────────────┐                       ┌───────────────────┐
  │ Forward LSTM Cell │                       │ Backward LSTM Cell│
  └───────────────────┘                       └───────────────────┘
            │                                           │
            ▼                                           ▼
      h_forward_t                                 h_backward_t
            │                                           │
            └───────────────┬───────────────────────────┘
                            │
                            ▼ Concatenate
                  h_t = [h_forward_t, h_backward_t]
```

**Inside a Single LSTM Cell** — each cell (Forward and Backward) runs 4 matrix multiplications at every step using the concatenated vector $[h_{t-1}, x_t]$:

1. **Forget Gate:** $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$
2. **Input Gate & Candidate State:** $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$, $\ \tilde{c}_t = \tanh(W_c \cdot [h_{t-1}, x_t] + b_c)$
3. **Cell State Update:** $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$
4. **Output Gate & Hidden State:** $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$, $\ h_t = o_t \odot \tanh(c_t)$

**Layer Stacking & Dropout Execution** — when stacking multiple layers (`num_layers=2`) with `dropout=0.3`:

```
Layer 2  ──►  Forward LSTM ──┐
         ──► Backward LSTM ──┼──► h_n[-2], h_n[-1] ──► [Concat] ──► FC Classifier
                   ▲         │
                   │ Dropout (p=0.3)
                   │ Zeroes 30% of activations during training
Layer 1  ──►  Forward LSTM ──┤
         ──► Backward LSTM ──┘
                   ▲
                   │
              Input Embeddings
```

**Backpropagation Through Time (BPTT)** — during the backward pass (training):
- Gradients flow **backward through time** along the cell state chain ($c_t \rightarrow c_{t-1}$).
- Because the cell state update uses **additive connections** ($c_t = f_t \odot c_{t-1} + \dots$) rather than purely matrix multiplications, gradients pass back across 200+ steps without exponentially shrinking (vanishing).

## 8. Data Preparation

The code below downloads the IMDB dataset, builds a simple vocabulary, and constructs the PyTorch `DataLoader`.

(Note: apostrophes are kept in the tokenizer — `[^a-z' ]+` instead of `[^a-z ]+` — so contractions like "wasn't" don't get mangled into "wasnt".)

In [ ]:
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim
from collections import Counter
import re

# 1. Load IMDB
print("Loading IMDB Dataset...")
dataset = load_dataset("stanfordnlp/imdb")
train_data = dataset['train'].shuffle(seed=42).select(range(25000))
test_data = dataset['test'].shuffle(seed=42).select(range(5000))

# 2. Simple Tokenizer & Vocabulary Builder
def tokenize(text):
    return re.sub(r"[^a-z' ]+", '', text.lower()).split()

print("Building vocabulary...")
all_words = [word for item in train_data for word in tokenize(item['text'])]
vocab = {word: i+2 for i, (word, _) in enumerate(Counter(all_words).most_common(10000))}
vocab['<PAD>'] = 0  # Index 0 ---> <PAD>: Used to pad shorter sequences in a batch so they all have equal length.
vocab['<UNK>'] = 1  # Index 1 ----> <UNK>: Represents "Unknown" words (words not in the top 10,000 vocabulary).

def encode(text):
    return [vocab.get(word, vocab['<UNK>']) for word in tokenize(text)][:250]  # Cap at 250 words

# 3. Collate function for DataLoader (Padding)
def collate_fn(batch):
    sequences = [torch.tensor(encode(item['text'])) for item in batch]
    labels = torch.tensor([item['label'] for item in batch]).float()
    padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)
    return padded_seqs, labels

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
print("DataLoader ready!")

**Why `collate_fn`?** A `DataLoader` grabs individual samples and needs to combine them into one batch tensor — trivial when every sample is the same shape, but reviews have different lengths (one might tokenize to 40 words, another to 250). `collate_fn` tells `DataLoader` how to combine a list of samples into a batch:

1. Encode each review's text into token IDs.
2. Collect labels into a float tensor.
3. `pad_sequence` finds the longest sequence *in that batch* and pads every shorter one with `0` (the `<PAD>` token) until they match — so a batch might come out `[32, 187]` if the longest review in it was 187 tokens, while the next batch could be `[32, 203]`.

Let's peek at one batch to confirm the shape:

In [ ]:
next(iter(train_loader))[0]

## 9. Training Loop

**Your Task**:
1. Initialize your device (GPU if available, else CPU), the model, the loss function (`BCEWithLogitsLoss`), and the optimizer (`Adam`).
2. Write the standard PyTorch training loop.

We also add a validation split, gradient clipping, and **early stopping** so training doesn't run for a fixed number of epochs regardless of whether the model has plateaued or started overfitting.

First, the validation loader and an `evaluate()` helper:

In [ ]:
val_loader = DataLoader(dataset['train'].select(range(2000, 2500)), batch_size=32, collate_fn=collate_fn)

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for texts, labels in loader:
            texts, labels = texts.to(device), labels.to(device)
            preds = (torch.sigmoid(model(texts)) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    model.train()
    return correct / total

- `model.eval()` disables dropout so predictions are deterministic while measuring accuracy; `model.train()` at the end re-enables it before the next training epoch.
- `preds = (torch.sigmoid(model(texts)) > 0.5).float()` converts each raw logit into a 0/1 label for every sample in the batch (not just the positive ones) so it can be compared elementwise against `labels`.
- `torch.no_grad()` skips gradient tracking since we're not backpropagating here — saves memory and compute.

Now the full training setup, with gradient clipping and early stopping:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

# 1. Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Model
embed_dim = 100
hidden_dim = 128
model = SentimentLSTM(len(vocab), embed_dim, hidden_dim).to(device)

# 3. Loss & optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-5)

print(f"Training on {device}...")

# 4. Training mode
model.train()

# Early stopping setup
best_val_acc = 0.0
best_model_state = None
patience = 3          # how many epochs to tolerate no improvement
patience_counter = 0
max_epochs = 20        # upper bound; early stopping will likely cut it short

for epoch in range(max_epochs):
    total_loss = 0

    # 5. Standard training loop
    for texts, labels in train_loader:
        # A. Move to device (labels as float for BCEWithLogitsLoss)
        texts = texts.to(device)
        labels = labels.to(device).float()

        # B. Zero the gradients
        optimizer.zero_grad()

        # C. Forward pass
        predictions = model(texts).squeeze()

        # D. Calculate loss
        loss = criterion(predictions, labels)

        # E. Backward pass
        loss.backward()

        # Gradient clipping — LSTMs (especially 2-layer bidirectional ones) are prone to exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        # F. Optimizer step
        optimizer.step()

        # G. Accumulate loss
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    val_acc = evaluate(model, val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}")

    # --- Early stopping logic ---
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())  # snapshot best weights
        patience_counter = 0
        print(f"  \u21b3 New best val acc: {best_val_acc:.4f} (saved)")
    else:
        patience_counter += 1
        print(f"  \u21b3 No improvement ({patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# Load back the BEST weights (not necessarily the last epoch's)
model.load_state_dict(best_model_state)
print(f"Loaded best model with val acc: {best_val_acc:.4f}")

**Why this beats a fixed epoch count:**

- `patience_counter` — one bad epoch doesn't mean the model has plateaued; loss curves are noisy. `patience=3` means "only stop if 3 epochs *in a row* fail to improve."
- `copy.deepcopy(model.state_dict())` — snapshots the actual weight values. Without `deepcopy`, `best_model_state` would just point at the *live* weights and get silently overwritten as training continues.
- Loading `best_model_state` at the end — training runs a few extra epochs after the peak (the cost of `patience`), and those extra epochs are likely overfitting. Loading the best-val-accuracy checkpoint instead of the final epoch avoids shipping an overfit model.
- `torch.nn.utils.clip_grad_norm_` caps the gradient norm before the optimizer step, preventing the exploding-gradient instability that stacked bidirectional LSTMs are prone to.

## 10. Testing & Inference

Let's see how the trained LSTM performs on completely unseen reviews.

**Your Task**: Write the inference logic to process raw strings, pass them through the trained network, and print the predicted sentiment.

In [ ]:
# 1. Set the model to evaluation mode
model.eval()

sample_reviews = [
    "This movie was an absolute masterpiece. The acting was phenomenal.",
    "Terrible. I wasted two hours of my life on this garbage.",
    "It started out a bit slow, but the ending was incredibly satisfying and emotional.",
    "Not good at all. The plot made no sense and the characters were boring.",
    "Great, another two hours of my life I'll never get back.",
]

print("--- LSTM Sentiment Predictions ---")

with torch.no_grad():
    for text in sample_reviews:
        # 2. Encode the text using the provided encode() function.
        encoded_list = encode(text)

        # 3. Convert the encoded list to a PyTorch tensor.
        tensor = torch.tensor(encoded_list)

        # 4. TRAP: tensor is currently 1D [seq_len]. The model expects [batch_size, seq_len].
        # Use .unsqueeze(0) to add a fake batch dimension of 1, and move it to the device.
        encoded_tensor = tensor.unsqueeze(0).to(device)

        # 5. Pass the tensor through the model to get the logit.
        logit = model(encoded_tensor)

        # 6. Apply torch.sigmoid() to the logit and extract the scalar float value using .item()
        probability = torch.sigmoid(logit).item()

        # 7. Determine the sentiment label
        sentiment = "POSITIVE" if probability > 0.5 else "NEGATIVE"

        print(f"\nReview: {text}")
        print(f"Prediction: {sentiment} ({probability:.4f})")

### A harder test: sarcasm

Sarcasm is a classic hard case for sentiment models — the surface words are positive but the meaning is negative. A small BiLSTM has no real mechanism to detect the contradiction between "loved" and "made zero sense"; it mostly reacts to word polarity.

In [ ]:
sarcastic_review = (
    "Oh wow, what a masterpiece. I especially loved how the plot made zero sense "
    "and the lead actor delivered every single line like he was reading it off a "
    "cereal box for the first time. Truly a triumph of modern cinema, if your goal "
    "was to make me question why I own eyes."
)

model.eval()
with torch.no_grad():
    encoded_tensor = torch.tensor(encode(sarcastic_review)).unsqueeze(0).to(device)
    logit = model(encoded_tensor)
    probability = torch.sigmoid(logit).item()
    sentiment = "POSITIVE" if probability > 0.5 else "NEGATIVE"

print(f"Review: {sarcastic_review}")
print(f"Prediction: {sentiment} ({probability:.4f})")